# OSRT v6 SFT-v3 — chat testing with token stats

Interactive testing of the **post-SFT** model (`osrt_v5_sft_v3_final.pt`, 800 steps on the 42K verified+Nemotron+smoltalk2 corpus, base = midtrain3).

Unlike the base-model notebook, this one speaks the **v6 chat contract**:
```
<|system|>{persona}<|user|>{your prompt}<|assistant|>  ← we send this
<|think|>reasoning<|/think|><|answer|>final answer<|/answer|>  ← model generates this
```

**Three modes**, using the exact personas the corpus was built with:
- **ON** — long reasoning in `<|think|>`, then the answer (54% of training rows)
- **OFF** — answer directly, empty/brief think (30%)
- **CHAT** — friendly conversational reply (15%)

The mode is chosen purely by the **system prompt** — same weights. If ON produces reasoning and OFF doesn't, the toggle transferred.

**Setup**: GPU runtime + `HF_TOKEN` in the Colab secrets sidebar (🔑). First generation compiles (several minutes); everything after is full speed.

In [ ]:
# ── 1. Setup: repo code + deps + HF auth ─────────────────────────────
import os, sys

BRANCH = "feat/sft-harvest"
if not os.path.isdir("/content/osrt"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/CodeHalwell/OSRT-605M-A269M.git /content/osrt
%pip -q install -U huggingface_hub transformers
sys.path.insert(0, "/content/osrt/src")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), f"| sm_{cap[0]}{cap[1]} | torch", torch.__version__)

In [ ]:
# ── 2. Load the SFT checkpoint ───────────────────────────────────────
CKPT = "osrt_v5_sft_v3_final.pt"   # or osrt_v5_sft_v3_step_{200,400,600,800}.pt
HF_REPO = "HallD/osrt-v6-ckpt"

# self-heal after a kernel restart (sys.path/env are per-kernel)
import os, sys
if "/content/osrt/src" not in sys.path:
    sys.path.insert(0, "/content/osrt/src")
if not os.environ.get("HF_TOKEN"):
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
import torch

from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer
from osrt.model import OSRTForCausalLM
from osrt.presets import build_config

path = hf_hub_download(HF_REPO, CKPT, repo_type="model")
# 65K v6 contract. The repo's tokenizer/ dir is a STALE 32K artifact.
tok = AutoTokenizer.from_pretrained("/content/osrt/v6_tokenizer_export")
assert len(tok) == 65536, f"wrong tokenizer: {len(tok)} tokens, expected 65536"
cfg = build_config(
    vocab_size=len(tok), real_vocab_size=len(tok),
    bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id, fused_cross_entropy_chunks=8,
)
device = torch.device("cuda")
model = OSRTForCausalLM(cfg).to(device)
sd = torch.load(path, map_location=device, weights_only=True)
sd = sd.get("model_state_dict", sd)
missing, unexpected = model.load_state_dict(sd, strict=False)
assert not missing and not unexpected, f"state mismatch: {missing[:3]} {unexpected[:3]}"
model.eval()

# The contract tokens must each be ONE token — the format is learned as
# single symbols, and we stop generation on <|/answer|>.
for t in ("<|system|>", "<|user|>", "<|assistant|>", "<|think|>", "<|/think|>",
          "<|answer|>", "<|/answer|>"):
    ids = tok.encode(t, add_special_tokens=False)
    assert len(ids) == 1, f"{t} is not a single token: {ids}"
END_ANSWER_ID = tok.encode("<|/answer|>", add_special_tokens=False)[0]

# eager smoke forward before compiling (readable traceback if the arch
# trips something, e.g. torch._grouped_mm on a new GPU)
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
    ids = torch.tensor([[tok.bos_token_id] + tok.encode("Hello", add_special_tokens=False)], device=device)
    model(ids)
print(f"loaded {CKPT} — contract tokens OK, smoke forward OK")

In [ ]:
# ── 3. Optimize + warmup (the slow cell — compile happens here) ──────
COMPILE = True
CUDA_GRAPHS = True
CACHE_IMPL = "static"   # "static" = speed mode | "latent" = original path

model.optimize_for_inference(compile_model=COMPILE, reduce_overhead=CUDA_GRAPHS)

import time
print("warmup (compiles on first call — several minutes)...")
t0 = time.perf_counter()
with torch.amp.autocast("cuda", dtype=torch.bfloat16):
    model.generate(ids, max_new_tokens=8, cache_impl=CACHE_IMPL)
    model.generate(ids, max_new_tokens=8, cache_impl=CACHE_IMPL)  # graph capture warm
torch.cuda.synchronize()
print(f"warm ({time.perf_counter() - t0:.0f}s). Full speed from here.")

In [ ]:
# ── 4. Chat helper: contract wrapping, think/answer parsing, stats ───
import re, time
from osrt.system_prompts import get_by_name

# The EXACT personas the v3 corpus was built with (domain-neutral pools —
# scripts/build_sft_v3_data.py). Swap the name to try a different one.
PERSONAS = {
    "on":   "minimal_format",    # also: concise_direct, reasoning_3shot,
                                 #   instruction_strict, verbose_teaching,
                                 #   casual_helpful, general_default
    "off":  "direct_concise",    # also: no_reasoning, assistant_plain,
                                 #   instruction_direct, chat_direct
    "chat": "chat_direct",
}

def chat(prompt: str, mode: str = "on", max_new_tokens: int = 512,
         temperature: float = 0.7, top_p: float = 0.95, top_k: int = 40,
         repetition_penalty: float = 1.05, persona: str | None = None,
         system: str | None = None, num_loops: int | None = None,
         stop_at_answer: bool = True) -> dict:
    """One turn through the v6 chat contract, instrumented.

    TTFT comes from a separate 1-token call (prefill + first decode step);
    the steady-state decode rate is derived from the remainder of the full
    call, so it reflects warm-path throughput."""
    sys_text = system if system is not None else get_by_name(persona or PERSONAS[mode])
    text = f"<|system|>{sys_text}<|user|>{prompt}<|assistant|>"
    ids = [tok.bos_token_id] + tok.encode(text, add_special_tokens=False)
    inp = torch.tensor([ids], device=device)
    kw = dict(temperature=temperature, top_p=top_p, top_k=top_k,
              repetition_penalty=repetition_penalty,
              eos_token_id=tok.eos_token_id, cache_impl=CACHE_IMPL,
              num_loops=num_loops)
    # Stop as soon as the answer block closes — the trained EOS should do
    # this anyway, so a large gap between the two is itself a finding.
    if stop_at_answer:
        kw["stop_token_ids"] = [END_ANSWER_ID]
    torch.cuda.reset_peak_memory_stats()

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        model.generate(inp, max_new_tokens=1, **kw)
    torch.cuda.synchronize(); ttft = time.perf_counter() - t0

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(inp, max_new_tokens=max_new_tokens, **kw)
    torch.cuda.synchronize(); total = time.perf_counter() - t0

    gen_ids = out[0, len(ids):].tolist()
    raw = tok.decode(gen_ids, skip_special_tokens=False)
    n = len(gen_ids)

    def _block(tag: str) -> str:
        m = re.search(rf"<\|{tag}\|>(.*?)(?:<\|/{tag}\|>|$)", raw, re.S)
        return m.group(1).strip() if m else ""

    think, answer = _block("think"), _block("answer")
    return {
        "mode": mode, "persona": persona or PERSONAS[mode],
        "think": think, "answer": answer, "raw": raw,
        "format_ok": bool(re.search(r"<\|answer\|>.*?<\|/answer\|>", raw, re.S)),
        "hit_cap": n >= max_new_tokens,
        "prompt_tokens": len(ids), "new_tokens": n,
        "think_chars": len(think), "answer_chars": len(answer),
        "ttft_ms": ttft * 1e3, "prefill_tps": len(ids) / ttft,
        "decode_tps": (n - 1) / max(1e-9, total - ttft) if n > 1 else float("nan"),
        "e2e_tps": n / total, "total_s": total,
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 2**30,
    }

def show(r: dict, show_think: bool = True) -> None:
    print(f"[{r['mode'].upper()} · {r['persona']}]")
    if show_think and r["think"]:
        print(f"\n\033[2m--- think ({r['think_chars']} chars) ---\n{r['think']}\033[0m")
    answer_text = r["answer"] or "(no answer block parsed — inspect r['raw'])"
    print(f"\n--- answer ---\n{answer_text}")
    flags = []
    if not r["format_ok"]: flags.append("BAD FORMAT")
    if r["hit_cap"]:       flags.append("HIT TOKEN CAP (no stop)")
    warn = ("  /!\\ " + " · ".join(flags)) if flags else ""
    print("\n" + "─" * 72)
    print(f"prompt {r['prompt_tokens']} tok | generated {r['new_tokens']} tok "
          f"(think {r['think_chars']}c / answer {r['answer_chars']}c) | "
          f"TTFT {r['ttft_ms']:.0f} ms | decode {r['decode_tps']:.1f} tok/s | "
          f"e2e {r['e2e_tps']:.1f} tok/s | {r['total_s']:.2f}s | "
          f"VRAM {r['peak_vram_gb']:.1f} GB" + warn)

show(chat("Natalia sold clips to 48 friends in April, and then she sold half as many clips in May. "
          "How many clips did she sell altogether in April and May?", mode="on"))

In [ ]:
# ── 5. The reasoning ON/OFF toggle on ONE prompt ─────────────────────
# Same weights, same question — only the system prompt differs. This is the
# clearest single test of whether the contract transferred.
Q = "A shop sells pens for £3 each. If I buy 7 pens and pay with a £25 note, how much change do I get?"

for m in ("on", "off", "chat"):
    print("=" * 72)
    show(chat(Q, mode=m, max_new_tokens=400))
    print()

In [ ]:
# ── 6. Interactive loop ──────────────────────────────────────────────
# Type a prompt. Prefix to switch mode:  "off: ..."  "chat: ..."  (default ON)
# Empty line or 'quit' exits.
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.7     # 0.0 = greedy
SHOW_THINK = True

while True:
    try:
        line = input("\nprompt> ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not line or line.lower() in {"quit", "exit"}:
        break
    mode = "on"
    for m in ("on", "off", "chat"):
        if line.lower().startswith(f"{m}:"):
            mode, line = m, line[len(m) + 1:].strip()
            break
    if not line:
        continue
    show(chat(line, mode=mode, max_new_tokens=MAX_NEW_TOKENS,
              temperature=TEMPERATURE), show_think=SHOW_THINK)

In [ ]:
# ── 7. GSM8K spot-check (preview of the GO/NO-GO gate) ───────────────
# NOT the official eval (that's the lm-eval-harness / sft_eval stage on the
# full test split) — a quick greedy read on N held-out test problems to see
# roughly where we stand. Gate: >=~15% -> GRPO is worth it; <10% -> rethink.
N_PROBLEMS = 20

import re as _re, time as _time
import pandas as pd
from huggingface_hub import hf_hub_download
from osrt.rewards import extract_numeric_answer

# Load the test split as a PARQUET FILE, not via load_dataset(). The dataset
# -module path does a HEAD on README.md through api/resolve-cache, which
# 504s during Hub gateway wobbles; the file-resolve endpoint keeps working.
def _load_gsm8k_test(tries: int = 4) -> pd.DataFrame:
    for k in range(tries):
        try:
            p = hf_hub_download("openai/gsm8k", "main/test-00000-of-00001.parquet",
                                repo_type="dataset")
            return pd.read_parquet(p)
        except Exception as e:
            if k == tries - 1:
                raise
            wait = 5 * 2**k
            print(f"  Hub error ({type(e).__name__}); retry {k+1}/{tries-1} in {wait}s")
            _time.sleep(wait)

df = _load_gsm8k_test()
rows = df.head(N_PROBLEMS).to_dict("records")
print(f"loaded {len(df)} GSM8K test problems; evaluating {len(rows)}\n")

correct, fmt_ok, caps, tps = 0, 0, 0, []
for i, row in enumerate(rows, 1):
    gold = _re.search(r"####\s*([\-0-9\.,]+)", row["answer"]).group(1).replace(",", "").strip()
    r = chat(row["question"], mode="on", max_new_tokens=512, temperature=0.0)
    pred = extract_numeric_answer(r["answer"] or r["raw"])
    hit = pred is not None and str(pred).replace(",", "").strip().rstrip(".") == gold
    correct += hit; fmt_ok += r["format_ok"]; caps += r["hit_cap"]; tps.append(r["decode_tps"])
    print(f"{i:>3}. {'PASS' if hit else 'fail'} pred={pred} gold={gold} "
          f"| think {r['think_chars']}c | {r['new_tokens']} tok"
          f"{' | CAP' if r['hit_cap'] else ''}")

n = len(rows)
print("\n" + "=" * 72)
print(f"GSM8K spot-check: {correct}/{n} = {100*correct/n:.0f}% correct")
print(f"format valid: {fmt_ok}/{n} | hit token cap: {caps}/{n} | "
      f"mean decode {sum(tps)/len(tps):.1f} tok/s")
print("NOTE: n=20 has a wide error bar (+/-~10pp). Treat as a smell test; the\n"
      "      full-split eval is the number that decides GRPO.")

## What to look for

**Format adherence** — `format_ok` should be ~100%. This was already working in SFT v1, so a failure here means something is wrong with loading, not learning.

**The ON/OFF toggle** — ON should produce substantive `<|think|>` content; OFF should produce little or none. If both look the same, the system prompt isn't steering and the contrast slices didn't take.

**Stopping** — `HIT TOKEN CAP` warnings mean the model isn't emitting EOS after `<|/answer|>`. The v3 corpus builder appends a real EOS label specifically to fix the SFT-v2 runaway, so caps appearing often is a regression worth reporting.

**Reasoning quality** — the honest question. v1's lesson was "format works, reasoning incoherent". Watch whether `<|think|>` actually *works the problem* (uses the numbers from the question, arrives somewhere) versus performing the shape of reasoning. Arithmetic slips with sound method are the good failure mode at 601M; confident nonsense is the bad one.

**Checkpoint comparison** — set `CKPT` in cell 2 to `osrt_v5_sft_v3_step_200.pt` / `_400` / `_600` to see whether quality was still improving at the end (informs whether more SFT steps would pay).

## Notes

- `chat(..., persona="reasoning_3shot")` overrides the persona; `system="..."` sends a completely custom system prompt (the model was trained to follow varied ones, so this is fair game).
- `temperature=0.0` for reproducible greedy output; the default 0.7 is more representative of normal use.
- `stop_at_answer=False` shows what the model does after closing the answer block — useful for diagnosing EOS behaviour.
- `num_loops=` (1–6) trades quality for speed. Untested for quality on the SFT model; probing only.
- Decode-speed reference (batch 1): A100 ~92 · H100 ~119 · B200 ~136 · RTX 6000 Pro ~112 tok/s.